In [ ]:
"""
6. Iterative Pruning — ResNet-50 / CIFAR-10
============================================
Iterative pruning progressively removes weights in multiple rounds,
measuring accuracy at each intermediate step.

Unlike one-shot pruning (which reaches the target sparsity in one step),
iterative pruning:
  1. Prune a small fraction of weights
  2. Evaluate accuracy
  3. Repeat until target sparsity reached

This tracks the pruning trajectory and allows comparison of:
  - Accuracy at each intermediate sparsity step
  - Degradation rate: gradual vs sudden accuracy drop
  - The "sweet spot" where accuracy starts collapsing

Rounds can be configured; each experiment tracks the full trajectory.

Output: 6_iterative_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "6_v2_iterative_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Iterative schedule: prune STEP% of REMAINING weights each round
# This defines a trajectory ending near MAX_SPARSITY
PRUNING_STEP    = 0.15    # prune 15% of remaining weights each round
MAX_SPARSITY    = 0.90    # stop when this overall sparsity reached
MAX_ACC_DROP    = 0.02
INFERENCE_RUNS  = 50


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning ───────────────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def prune_one_step(model, step_amount):
    """Prune step_amount fraction of CURRENT non-zero weights globally."""
    # Compute threshold on current non-zero weights only
    all_weights = torch.cat([
        m.weight.data[m.weight.data != 0].abs().view(-1)
        for _, m in model.named_modules()
        if isinstance(m, (nn.Conv2d, nn.Linear))
    ])
    if all_weights.numel() == 0:
        return model

    k = max(1, int(all_weights.numel() * step_amount))
    threshold = torch.kthvalue(all_weights, k).values.item()

    with torch.no_grad():
        for _, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                mask = (module.weight.data.abs() > threshold).float()
                module.weight.data *= mask

    return model

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(10): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(3): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(10): cpu_m(dummy)
    return (time.perf_counter() - t0) / 10 * 1000

def compute_flops(model):
    """Total theoretical FLOPs — shape-based, no weight values needed."""
    flop_counts = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                _, c_out, oH, oW = out.shape
                flop_counts[name] = 2 * module.in_channels * c_out * module.kernel_size[0] * module.kernel_size[1] * oH * oW
            elif isinstance(module, nn.Linear):
                flop_counts[name] = 2 * module.in_features * module.out_features
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(torch.randn(1, 3, 32, 32).to(DEVICE))

    for h in hooks: h.remove()
    return sum(flop_counts.values()), flop_counts

def compute_active_flops(model, flop_counts):
    """FLOPs scaled by per-layer weight density (approximates sparse compute)."""
    active = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in flop_counts:
            density = (module.weight != 0).float().mean().item()
            active += flop_counts[name] * density
    return int(active)

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*62}")
    print("  6. Iterative Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Step: {int(PRUNING_STEP*100)}% per round  |  Max: {int(MAX_SPARSITY*100)}%")
    print(f"{'='*62}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()
    base_disk = disk_size_mb(model)
    base_flops, flop_counts = compute_flops(model)

    # Iterative pruning: accumulate trajectory
    current   = copy.deepcopy(model)
    trajectory = []   # one entry per round
    best_model = None
    best_score = -1.0
    best_sp    = None
    round_num  = 0

    while True:
        round_num += 1
        print(f"\n  Round {round_num}: pruning {int(PRUNING_STEP*100)}% of remaining weights...")
        current   = prune_one_step(current, PRUNING_STEP)
        actual_sp = real_sparsity(current)

        print(f"  Cumulative sparsity: {actual_sp*100:.1f}%")

        total_p, active_p = count_params(current)
        metrics  = evaluate(current, loader)
        inf_gpu  = measure_gpu_ms(current)
        inf_cpu  = measure_cpu_ms(current)
        acc_drop = baseline["accuracy"] - metrics["accuracy"]
        sp_mb    = sparse_size_mb(current)
        dk_mb    = disk_size_mb(current)

        print(f"  Acc: {metrics['accuracy']:.4f} (drop: {acc_drop:+.4f}) | "
              f"Sparse MB: {sp_mb:.1f} | Disk: {dk_mb:.1f}")

        entry = {
            "round"                      : round_num,
            "pruning_step_fraction"      : PRUNING_STEP,
            "cumulative_sparsity"        : round(actual_sp, 4),
            "accuracy"                   : round(metrics["accuracy"], 6),
            "precision"                  : round(metrics["precision"], 6),
            "recall"                     : round(metrics["recall"], 6),
            "f1"                         : round(metrics["f1"], 6),
            "accuracy_drop"              : round(acc_drop, 6),
            "params_total"               : total_p,
            "params_active"              : active_p,
            "size_sparse_theoretical_mb" : round(sp_mb, 4),
            "size_disk_mb"               : round(dk_mb, 4),
            "disk_saved_mb"              : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
            "inference_cpu_ms"           : round(inf_cpu, 4),
            "flops_total"   : int(base_flops),
            "flops_active"  : compute_active_flops(current, flop_counts),
            "flops_reduction_pct": round((1 - compute_active_flops(current, flop_counts) / base_flops) * 100, 2)

        }
        trajectory.append(entry)

        if acc_drop <= MAX_ACC_DROP:
            score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score, best_model, best_sp = score, copy.deepcopy(current), actual_sp

        if actual_sp >= MAX_SPARSITY:
            print(f"\n  Reached max sparsity ({MAX_SPARSITY*100:.0f}%). Stopping.")
            break
        if round_num > 30:  # safety cap
            print(f"\n  Max rounds reached. Stopping.")
            break

    # Summarize final sparsity buckets (for comparison with other methods)
    # Map trajectory entries to standard sparsity checkpoints
    CHECKPOINTS = [0.6229] # 0.30, 0.50, 0.70, 0.80, 0.90
    checkpointed = []
    for target in CHECKPOINTS:
        # Find closest round
        closest = min(trajectory, key=lambda x: abs(x["cumulative_sparsity"] - target))
        checkpointed.append({**closest, "checkpoint_target": target})

    report = {
        "method"              : "iterative_pruning",
        "description"         : f"Iterative L1 magnitude pruning: {int(PRUNING_STEP*100)}% of remaining weights per round",
        "pruning_granularity" : "weight (unstructured, global L1)",
        "pruning_step"        : PRUNING_STEP,
        "total_rounds"        : round_num,
        "baseline"            : baseline,
        "device"              : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_sparsity"       : round(best_sp, 4) if best_sp else None,
        "trajectory"          : trajectory,        # full round-by-round log
        "checkpointed_results": checkpointed,       # snapped to standard levels
        "notes": {
            "trajectory"      : "Full pruning trajectory: accuracy at each incremental round",
            "advantage"       : "Reveals where accuracy begins to drop, not just final value",
            "vs_one_shot"     : "Compare with 7_oneshot_Pruning.json to see iterative vs one-shot difference",
            "step_formula"    : f"Each round removes {int(PRUNING_STEP*100)}% of REMAINING non-zero weights (not of total)",
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")
    print(f"  Total rounds: {round_num}  |  Final sparsity: {trajectory[-1]['cumulative_sparsity']*100:.1f}%")


if __name__ == "__main__":
    main()


  6. Iterative Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Step: 15% per round  |  Max: 90%


  Round 1: pruning 15% of remaining weights...
  Cumulative sparsity: 15.0%
  Acc: 0.9321 (drop: -0.0001) | Sparse MB: 76.3 | Disk: 90.0

  Round 2: pruning 15% of remaining weights...
  Cumulative sparsity: 27.7%
  Acc: 0.9321 (drop: -0.0001) | Sparse MB: 64.9 | Disk: 90.0

  Round 3: pruning 15% of remaining weights...
  Cumulative sparsity: 38.6%
  Acc: 0.9320 (drop: +0.0000) | Sparse MB: 55.2 | Disk: 90.0

  Round 4: pruning 15% of remaining weights...
  Cumulative sparsity: 47.8%
  Acc: 0.9321 (drop: -0.0001) | Sparse MB: 46.9 | Disk: 90.0

  Round 5: pruning 15% of remaining weights...
  Cumulative sparsity: 55.6%
  Acc: 0.9320 (drop: +0.0000) | Sparse MB: 39.9 | Disk: 90.0

  Round 6: pruning 15% of remaining weights...
  Cumulative sparsity: 62.3%
  Acc: 0.9322 (drop: -0.0002) | Sparse MB: 34.0 | Disk: 90.0

  Round 7: pruning 15% of remaining weights...
  Cumulative sparsity: 6